# Static imaging with NUFFT

This notebook follows Kine’s static imaging example: it loads data, prepares the operator, and trains the neural field. First, install this checkout in your Kine environment (`python -m pip install --no-deps -e .` from the repository root), as well as `jax-finufft`. 

See [installation](../docs/installation.rst), then run from the repository root or `notebooks/`. 

In [ ]:
%matplotlib inline
from pathlib import Path
from collections import OrderedDict
import numpy as np
import matplotlib.pyplot as plt
import ehtim as eh
import jax
import jax.numpy as jnp
import optax
import kine.obsdata as ob
import kine.model as mo
import kine.trainer as tr
import kine.utils as ut
import kine.video as vi

## 1. Load the bundled M87 observation

`NPIX` controls sampling. These small grid and iteration settings are for a quick example, and can be extended for convergence. 

In [ ]:
TTYPE = "nufft"  # Change to "direct" to use the dense A matrix.
NPIX = 32
INIT_STEPS = 300
FIT_STEPS = 200
FOV_UAS = 160

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent
path = root / "data/eht_m87_2017-04-10.uvfits"
obs = ob.Obsdata.load_uvfits(str(path)).avg_coherent(600)
obs = obs.add_fractional_noise(0.01)
flux_proxy = obs.get_zbl()
fov = FOV_UAS * eh.RADPERUAS
prior = eh.image.make_square(obs, NPIX, fov)
grid = ut.get_grid(NPIX, NPIX)

## 2. Select the Fourier operator

The target and sigma arrays are unchanged. For NUFFT, `A` includes the coordinates and pixel factors.

In [ ]:
data = {}
for dtype in ("cphaseI", "logcampI"):
    target, sigma, A = ob.Obsdata.get_data(obs, dtype, prior, ttype=TTYPE)
    data[dtype] = {"target": target, "sigma": sigma, "A": A}
    print(dtype, "target:", target.shape, "sigma:", sigma.shape)

## 3. Initialize and fit

As in the static example, we first fit the blurred disk. We then fit the closure phase, log closure amplitude, and a total-flux proxy from the shortest baseline.

In [ ]:
model = mo.NeuralField(posenc_deg=(0, 0), depth=2, width=32,
                       outshift=0, scaling_i=1 / NPIX**2)
variables = model.init(jax.random.PRNGKey(163), grid, train=True)
state = tr.Trainer.create(
    apply_fn=model.apply, params=variables["params"],
    batch_stats=variables["batch_stats"],
    tx=optax.adamax(optax.piecewise_constant_schedule(0.01, {INIT_STEPS: 0.1})),
)
start = vi.Image(NPIX, fov, obs.ra, obs.dec, INIT_STEPS)
start.add_tophat(flux_proxy, {"fwhm": 80, "blur": 20, "posx": 0, "posy": 0})
for _ in range(INIT_STEPS):
    loss, _, _, state = tr.Trainer.train_step(OrderedDict(
        state=state, grid=grid, init_arr=start.iarr))

In [ ]:
history = {name: [] for name in (*data, "lcurve")}
for _ in range(FIT_STEPS):
    loss, terms, _, state = tr.Trainer.train_step(OrderedDict(
        state=state, grid=grid, data=data, lcurve=jnp.array([flux_proxy])))
    for name in history:
        history[name].append(float(terms[name]))
result = vi.Image(NPIX, fov, obs.ra, obs.dec, FIT_STEPS)
result.from_state(state, grid)
print({name: values[-1] for name, values in history.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout="constrained")
extent = [FOV_UAS / 2, -FOV_UAS / 2, -FOV_UAS / 2, FOV_UAS / 2]
im = axes[0].imshow(np.asarray(result.iarr[..., 0]), origin="lower",
                    extent=extent, cmap="inferno", interpolation="nearest")
axes[0].set(xlabel=r"RA offset ($\mu$as)", ylabel=r"Dec offset ($\mu$as)")
fig.colorbar(im, ax=axes[0], label="Flux per pixel (Jy)")
for name, values in history.items():
    axes[1].semilogy(np.arange(1, FIT_STEPS + 1), np.maximum(values, 1e-15), label=name)
axes[1].set(xlabel="Fitting iteration", ylabel="Native loss term")
axes[1].legend()
plt.show()

Try changing only `TTYPE` to `"direct"`, then restarting and running all cells to compare the final loss terms. 

Both methods predict measurements from the same pixel image. Since the NUFFT approximates that calculation, small differences are expected. Note that more pixels cost more memory with the dense matrix, and that the existing video interface is unchanged here.